In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import kagglehub
import os, pandas as pd
from joblib import dump, load

/home/rohnak.agarwal/projects/ml-practice/venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Download latest version
path = kagglehub.dataset_download("dragonheir/logistic-regression")

print("Path to dataset files:", path)

Path to dataset files: /home/rohnak.agarwal/.cache/kagglehub/datasets/dragonheir/logistic-regression/versions/1


In [3]:
df = pd.read_csv(os.path.join(path, "Social_Network_Ads.csv"))
df.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [4]:
df.drop(["User ID"], axis=1, inplace=True)
df.head()

,Gender,Age,EstimatedSalary,Purchased
0,Male,19,19000,0
1,Male,35,20000,0
2,Female,26,43000,0
3,Female,27,57000,0
4,Male,19,76000,0


In [5]:
X = df[["Gender", "Age", "EstimatedSalary"]]
y = df["Purchased"]

In [6]:
X.head()

,Gender,Age,EstimatedSalary
0,Male,19,19000
1,Male,35,20000
2,Female,26,43000
3,Female,27,57000
4,Male,19,76000


In [7]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Purchased, dtype: int64

In [8]:
# Separate categorical and numerical columns
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

print(cat_cols, num_cols)

Index(['Gender'], dtype='object') Index(['Age', 'EstimatedSalary'], dtype='object')


In [9]:
# One-hot encode categorical columns
X_encoded = pd.get_dummies(X, columns=cat_cols, dtype=float, drop_first=True)
X_encoded.head()

,Age,EstimatedSalary,Gender_Male
0,19,19000,1.0
1,35,20000,1.0
2,26,43000,0.0
3,27,57000,0.0
4,19,76000,1.0


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=327
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

(320, 3) (80, 3) (320,) (80,)


In [11]:
!mkdir -p ./model

In [12]:
dump(scaler, "./model/stdscaler.joblib")
pd.DataFrame(
    list(zip(scaler.feature_names_in_, scaler.mean_, scaler.scale_)),
    columns=["param", "mean", "scale"],
)

,param,mean,scale
0,Age,38.128125,10.562869
1,EstimatedSalary,69393.750000,34544.553998


In [13]:
scaler = load("./model/stdscaler.joblib")
pd.DataFrame(
    list(zip(scaler.feature_names_in_, scaler.mean_, scaler.scale_)),
    columns=["param", "mean", "scale"],
)

,param,mean,scale
0,Age,38.128125,10.562869
1,EstimatedSalary,69393.750000,34544.553998


In [14]:
!mkdir -p ./data/

In [15]:
X_train.to_csv("./data/x_train.csv", index=0)
X_test.to_csv("./data/x_test.csv", index=0)
y_train.to_csv("./data/y_train.csv", index=0)
y_test.to_csv("./data/y_test.csv", index=0)